# Legal Dataset Preparation

Download the source CSV, fetch each linked PDF, extract its text, and save a training-ready dataset.

In [3]:
from io import BytesIO
from pathlib import Path

import pandas as pd
import pypdfium2 as pdfium
import requests

DATASET_URL = "https://docs.google.com/spreadsheets/d/11o7R3TRtREbDcxcbCMUERu5WRZsgLZtTaRFUhr4jLtk/export?format=csv"
OUTPUT_CSV = Path("prepared_dataset.csv")
MAX_RECORDS = False

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

In [4]:
def download_pdf_bytes(url: str, timeout: int = 30) -> bytes:
    """Download a PDF and return its binary content."""
    response = requests.get(url, timeout=timeout)
    response.raise_for_status()
    return response.content

def extract_pdf_text(pdf_binary: bytes) -> str:
    """Extract text from PDF bytes using pypdfium2."""
    pdf = pdfium.PdfDocument(BytesIO(pdf_binary))
    pages = []
    for page_index in range(len(pdf)):
        page = pdf.get_page(page_index)
        text_page = page.get_textpage()
        pages.append(text_page.get_text_range())
        text_page.close()
        page.close()
    pdf.close()
    return "\n".join(pages)

def build_document_text(url: str) -> str:
    try:
        pdf_bytes = download_pdf_bytes(url)
        return extract_pdf_text(pdf_bytes)
    except Exception as exc:
        print(f"[!] Failed to process {url}: {exc}")
        return ""

In [5]:
raw_df = pd.read_csv(DATASET_URL)
if MAX_RECORDS:
    raw_df = raw_df.head(MAX_RECORDS)

rows = []
for idx, row in raw_df.iterrows():
    title = str(row.get("Title", "")).strip()
    abstract = str(row.get("Abstract", "")).strip()
    pdf_url = str(row.get("URL", "")).strip()

    if not pdf_url:
        print(f"[!] Missing PDF URL for row {idx}")
        document_text = ""
    else:
        document_text = build_document_text(pdf_url)

    rows.append({
        "Title": title,
        "Abstract": abstract,
        "DocumentText": document_text,
        "SourceURL": pdf_url,
    })

prepared_df = pd.DataFrame(rows)
prepared_df['DocumentText'] = prepared_df['DocumentText'].fillna('')
prepared_df['DocumentLength'] = prepared_df['DocumentText'].str.len()

print(f"Rows processed: {len(prepared_df)}")
print(prepared_df[['Title', 'DocumentLength']].head())

[!] Failed to process https://usableprivacy.org/data: Failed to load document (PDFium: Data format error).
[!] Failed to process https://usableprivacy.org/data: Failed to load document (PDFium: Data format error).
[!] Failed to process https://aclanthology.org/2023.nllp-1.24/: Failed to load document (PDFium: Data format error).
[!] Failed to process https://archive.ics.uci.edu/dataset/239/legal+case+reports: Failed to load document (PDFium: Data format error).
[!] Failed to process https://dl.acm.org/doi/10.1145/3322640.3326728: 403 Client Error: Forbidden for url: https://dl.acm.org/doi/10.1145/3322640.3326728
[!] Failed to process https://www.cst.cam.ac.uk/research/srg/projects/law: Failed to load document (PDFium: Data format error).
[!] Failed to process https://arxiv.org/abs/2104.08671: Failed to load document (PDFium: Data format error).
[!] Failed to process https://case.law/: Failed to load document (PDFium: Data format error).
[!] Failed to process https://www.researchgate.ne

In [6]:
filtered_df = prepared_df[prepared_df['DocumentLength'] >= 100].copy()
filtered_df.drop(columns=['DocumentLength'], inplace=True)
filtered_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(filtered_df)} rows to {OUTPUT_CSV}")
filtered_df.head()

Saved 7 rows to prepared_dataset.csv


,Title,Abstract,DocumentText,SourceURL
0,A Corpus of eRulemaking User Comments for Meas...,eRulemaking is a means for government agencies...,A Corpus of eRulemaking User Comments for Meas...,https://facultystaff.richmond.edu/~jpark/paper...
1,A Dataset for Statutory Reasoning in Tax Law E...,Legislation can be viewed as a body of prescri...,A Dataset for Statutory Reasoning in Tax Law E...,https://ceur-ws.org/Vol-2645/paper5.pdf
19,Legal Linking: Citation Resolution and Suggest...,This paper describes a dataset and baseline sy...,Proceedings of the Natural Legal Language Proc...,https://aclanthology.org/W19-2205.pdf
22,LegalDiscourse: Interpreting When Laws Apply a...,While legal AI has made strides in recent year...,Proceedings of the 2024 Conference of the Nort...,https://aclanthology.org/2024.naacl-long.472.pdf
38,Sentence Boundary Detection in Adjudicatory De...,We report results of an effort to enable compu...,Sentence Boundary Detection in\r\nAdjudicatory...,https://www.atala.org/sites/default/files/2-%2...
